# Lab 1 - Exercise 3.1: Improving Fine-tuning Performance

Questo notebook continua il lavoro dei notebook 02 e 02b. Nell'Exercise 1.3 abbiamo visto che allenare solo la testa finale della ResNet-18 non basta a sfruttare bene il backbone pre-addestrato.

In questo esercizio organizziamo una sequenza di prove riproducibili per migliorare il fine-tuning, mantenendo il notebook essenziale e coerente con la pipeline.

---
### Exercise 3.1 (Easy): Improving Fine-tuning Performance

In this exercise you are asked to iterate on the fine-tuning experiment performed in Exercise 1.3 in order to squeeze the best performance possible out of the model.

What can we do:

- Use a more powerful model?
- More aggressive data augmentation?
- Selective layer training?
- Something else?

L'idea scelta qui e' **selective layer training**: invece di allenare solo la `fc`, sblocchiamo anche `layer4`, cioe' l'ultimo blocco residuo della ResNet-18. E' un compromesso semplice: adattiamo le feature piu' alte al dominio GTSRB senza riaddestrare tutta la rete.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.evaluate import history_to_frame
from dla_lab1.models import build_classifier, count_parameters

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo training lunghi automaticamente.
RUN_TRAINING = False

print(f"Project root: {ROOT}")


## Punto di partenza: baseline Exercise 1.3

La baseline head-only e' il riferimento metodologico: ResNet-18 pre-addestrata, backbone congelato e solo classificatore finale addestrabile.

In questo notebook non inseriamo valori numerici precompilati. Il confronto deve essere fatto con risultati prodotti dalla pipeline corrente o con artifact generati localmente.


In [ ]:
baseline_context = pd.DataFrame([
    ["Exercise 1.2", "ResNet-18 feature extractor + SVM", "baseline stabile senza fine-tuning"],
    ["Exercise 1.3", "ResNet-18 frozen backbone + linear fc", "fine-tuning head-only"],
], columns=["Reference", "Setup", "Role"])

baseline_context


Nota: questa sezione mantiene il metodo sperimentale in forma riproducibile. I risultati finali devono essere prodotti dalle run della pipeline corrente, cosi' ogni valore mostrato nel notebook resta verificabile.


## Esperimenti possibili per Exercise 3.1

Le idee da provare sono tre:

- sbloccare `layer4` mantenendo congelato il resto del backbone;
- aggiungere augmentation moderata;
- combinare `layer4` trainabile e augmentation.

La tabella seguente elenca le configurazioni disponibili, senza riportare risultati precompilati.


In [ ]:
improvement_candidates = pd.DataFrame([
    ["ex3_1_layer4_unfrozen", "layer4 + fc", "none", "verifica effetto dello sblocco di layer4"],
    ["ex3_1_head_only_aggressive_aug", "fc only", "aggressive", "verifica se augmentation da sola basta"],
    ["ex3_1_layer4_aggressive_aug", "layer4 + fc", "aggressive", "augmentation forte con layer4 trainabile"],
    ["ex3_1_layer4_conservative_aug", "layer4 + fc", "conservative", "augmentation moderata con layer4 trainabile"],
    ["ex3_1_layer4_spatial_aug", "layer4 + fc", "spatial", "augmentation geometrica leggera"],
], columns=["Experiment", "Trainable layers", "Augmentation", "Purpose"])

improvement_candidates


## Analisi attesa

L'ipotesi principale e' che sbloccare `layer4` migliori la baseline head-only, perche' permette alle feature piu' alte della ResNet-18 di adattarsi ai segnali stradali.

L'augmentation va usata con attenzione: trasformazioni troppo aggressive possono creare segnali poco realistici. Per questo manteniamo una variante conservativa e una variante solo spaziale, entrambe spiegabili all'esame.


## Risultati della pipeline corrente

Questa sezione non contiene numeri scritti a mano. Se hai gia' eseguito una run con la pipeline corrente, il notebook prova a leggere `summary.json` dalla cartella `artifacts/runs`. Se non trova artifact, mostra solo un messaggio operativo.


In [ ]:
candidate_runs = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
]

run_rows = []
for run_name in candidate_runs:
    summary_path = ROOT / "artifacts" / "runs" / run_name / "summary.json"
    if summary_path.exists():
        summary = pd.read_json(summary_path, typ="series")
        run_rows.append([
            run_name,
            summary.get("best_epoch", None),
            summary.get("best_val_acc", None),
            summary_path,
        ])

if run_rows:
    current_ex3_results = pd.DataFrame(
        run_rows,
        columns=["Experiment", "Best Epoch", "Best Val Accuracy", "Summary Path"],
    ).sort_values("Best Val Accuracy", ascending=False)
    display(current_ex3_results)
else:
    print("Nessun summary.json trovato per le run candidate. Esegui una run per generare risultati nel notebook corrente.")


## Configurazioni nella nuova pipeline

Le configurazioni utili per Exercise 3.1 sono ora in `config/config.yaml`. Questo permette di rilanciare le prove senza riscrivere celle lunghe nel notebook.

In [ ]:
exercise_31_experiments = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
]

config_rows = []
for name in exercise_31_experiments:
    exp = experiment_config(config, name)
    config_rows.append([
        name,
        exp["model"]["name"],
        exp["model"]["unfreeze_layer4"],
        exp["experiment"].get("augmentation", "none"),
        exp["training"]["optimizer"],
        exp["training"]["loss"],
        exp["training"]["learning_rate"],
        exp["training"]["epochs"],
        batch_size_for(config, exp["experiment"]["batch_size_key"]),
    ])

pd.DataFrame(config_rows, columns=[
    "Experiment", "Model", "Unfreeze layer4", "Augmentation", "Optimizer",
    "Loss", "Learning rate", "Epochs", "Batch size"
])

## Controllo del modello selezionato

La variante selezionata per la run principale e' `ex3_1_layer4_conservative_aug`: ResNet-18 con backbone congelato, `layer4` e `fc` trainabili, e augmentation conservativa.

Questa configurazione e' un compromesso semplice da spiegare: aumenta la capacita' di adattamento rispetto alla baseline head-only, ma evita il fine-tuning completo di tutta la rete.


In [ ]:
SELECTED_EXPERIMENT = "ex3_1_layer4_conservative_aug"
selected_cfg = experiment_config(config, SELECTED_EXPERIMENT)

model = build_classifier(
    model_name=selected_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=selected_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(selected_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(selected_cfg["model"].get("unfreeze_layer4", False)),
)

parameter_summary = pd.DataFrame([count_parameters(model)])
parameter_summary["trainable_ratio"] = parameter_summary["trainable"] / parameter_summary["total"]
parameter_summary

Il numero di parametri trainabili e' molto piu' alto della baseline head-only, ma resta inferiore al fine-tuning completo. Questo e' il compromesso principale dell'esercizio: piu' adattamento al dataset senza aggiornare tutta la rete.

## Cella opzionale per rilanciare il miglior esperimento

La cella seguente e' disattivata di default. Serve solo se vuoi rieseguire la variante selezionata nella nuova pipeline.

In [ ]:
if RUN_TRAINING:
    result = run_finetuning(config, SELECTED_EXPERIMENT, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare Exercise 3.1.")


## Funzioni usate

Le funzioni principali sono commentate nei file `.py`:

- `experiment_config`: costruisce la configurazione completa dell'esperimento;
- `batch_size_for`: legge il batch size adatto dalla sezione hardware;
- `build_classifier`: crea ResNet-18 e sblocca `layer4` quando richiesto;
- `count_parameters`: controlla quanti parametri sono totali e trainabili;
- `run_finetuning`: esegue DataLoader, modello, training, checkpoint e artifact locali;
- `build_transforms`: applica preprocessing e augmentation scelta dal config.

## Conclusione Exercise 3.1

L'esercizio e' impostato in modo coerente con la richiesta: partiamo dalla baseline head-only, proponiamo miglioramenti incrementali, selezioniamo una configurazione spiegabile e la rendiamo rilanciabile dalla pipeline.

I risultati numerici devono essere prodotti dalla run corrente oppure letti dagli artifact generati localmente. In questo modo il notebook non dipende da tabelle compilate manualmente.

Quando esegui la run selezionata, il commento principale da fare riguarda il rapporto tra training accuracy e validation accuracy: se il training cresce molto piu' della validation, significa che il modello sta adattandosi bene al training set ma resta presente overfitting. Questo e' un punto importante da discutere all'esame.
